# Experiment 1 on Kaggle: CUB-200-2011 (GCD discovery + frozen CLIP grounding)

**Inputs.** None needed: the official CUB archive is downloaded from data.caltech.edu (MD5-checked)
and the DINO checkpoint from dl.fbaipublicfiles.com. GPU T4, Internet on.

**Resuming.** Every epoch writes a full checkpoint under `/kaggle/working/e1_runs`. If a session
ends before 200 epochs, open this notebook again, *Add Input → Notebook output* of the previous
version, and run again: section 2 copies the previous `e1_runs` back and training continues from
the last completed epoch (same configuration and manifest, or it refuses). CUB needs about one
session; the benchmark in section 5 gives the measured estimate.

## 1. Code and environment

In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path

DATASET = "cub"
REPO, REF = "eylulpelinkilic/generalized-category-discovery", "main"   # set REF to a commit to pin
TRAIN_SEED = 0              # 0 is the primary seed; run 1 and 2 as separate notebook copies
TRAIN_HOURS = 10.5         # training budget per session (Kaggle stops at 12 h)
CLUSTER_SEEDS = [0, 1, 2]
RUN_BENCHMARK = True        # once per dataset; skipped if a benchmark file exists
RUN_ESTIMATED_K = True
RUN_VITB16 = True           # CLIP ViT-B/16 robustness comparison after the primary ViT-B/32
WORK = Path("/kaggle/working")
CODE = WORK / "generalized-category-discovery"
EXP = CODE / "Experiment1_GCD_CLIP_Baseline"
RUNS = WORK / "e1_runs"
STATUS, T0 = {}, time.time()


def run(cmd, cwd=None, allow_fail=False):
    print(f"$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or (EXP if EXP.exists() else WORK), text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    tail = []
    for line in p.stdout:
        print(line, end="", flush=True)
        tail = (tail + [line.rstrip()])[-12:]
    code = p.wait()
    if code and not allow_fail:
        raise RuntimeError(f"exit {code}: {cmd}\n" + "\n".join(tail))
    return code


def step(name, fn, needs=()):
    missing = [n for n in needs if STATUS.get(n) != "ok"]
    if missing:
        STATUS[name] = f"skipped: needs {missing}"
        return
    try:
        fn()
        STATUS[name] = "ok"
    except Exception as exc:
        STATUS[name] = f"failed: {exc}"
        print(f"!! {name} failed: {exc}")


if not CODE.exists():
    run(f"git clone -q https://github.com/{REPO}.git {CODE}", cwd=WORK)
run(f"git fetch -q origin {REF} && git checkout -q FETCH_HEAD", cwd=CODE)
COMMIT = subprocess.run(["git", "rev-parse", "HEAD"], cwd=CODE, capture_output=True, text=True).stdout.strip()
print("code commit", COMMIT)
run("pip install -q open_clip_torch==3.3.0 ftfy regex")
run("pip install -q git+https://github.com/openai/CLIP.git", allow_fail=True)   # reference for verify-clip
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "Settings -> Accelerator: GPU T4"

## 2. Restore the previous session (if its output is attached)

In [ ]:
def restore_previous():
    """Copy e1_runs/<dataset> from an attached previous notebook output (if any)."""
    for depth in range(5):
        for cand in Path("/kaggle/input").glob("*/" * depth + f"e1_runs/{DATASET}"):
            target = RUNS / DATASET
            if not target.exists():
                print("restoring previous outputs from", cand)
                shutil.copytree(cand, target)
            return str(cand)
    return None


PREVIOUS = restore_previous()
print("previous outputs:", PREVIOUS)

## 3. Data location

In [ ]:
def locate_cub():
    for depth in range(5):
        for cand in Path("/kaggle/input").glob("*/" * depth + "CUB_200_2011/images.txt"):
            return cand.parent
    sys.path.insert(0, str(EXP))
    from e1.sources import fetch_cub
    return Path(fetch_cub("/tmp/cub")["path"])


IMAGE_ROOT = locate_cub()
SETS = f"--set paths.runs_root={RUNS} --set dataset.root={IMAGE_ROOT.parent} --set dataset.image_root={IMAGE_ROOT}"

## 4–7. Manifest, weights, benchmark, training (time-budgeted), and every downstream stage once training has reached its final epoch

In [ ]:
CFG = f"--config configs/{DATASET}.yaml"
WEIGHTS = Path("/tmp/weights")
DINO = WEIGHTS / "dino_vitbase16_pretrain.pth"
E1 = f"python -m e1 {CFG} {SETS} --set gcd_train.dino_checkpoint={DINO}"


def prepare():
    if (RUNS / DATASET / "data" / "manifest.csv").exists():
        print("manifest exists (immutable); checking it loads")
        run(f"{E1} report")
        return
    run(f"{E1} prepare-data --upstream-check")


def weights():
    run(f"{E1} fetch-dino --dest {WEIGHTS}")


def bench():
    if list((RUNS / DATASET / "gcd").glob("benchmark_*.json")):
        print("benchmark exists")
        return
    run(f"{E1} benchmark --train-seed {TRAIN_SEED} --iters 20 --amp-modes fp32,amp")


def train():
    left = TRAIN_HOURS - (time.time() - T0) / 3600
    run(f"{E1} train --train-seed {TRAIN_SEED} --time-budget-hours {max(0.2, left):.2f}")


def finished():
    return any((RUNS / DATASET / "gcd").glob(f"seed{TRAIN_SEED}_*/checkpoints/final.pt"))


def downstream():
    run(f"{E1} extract-gcd --train-seed {TRAIN_SEED}")
    for c in CLUSTER_SEEDS:
        run(f"{E1} cluster --train-seed {TRAIN_SEED} --k-mode known --cluster-seed {c}")
    if RUN_ESTIMATED_K:
        run(f"{E1} cluster --train-seed {TRAIN_SEED} --k-mode estimated --cluster-seed 0")
    clips = ["vitb32"] + (["vitb16"] if RUN_VITB16 else [])
    for clip in clips:
        run(f"{E1} verify-clip --clip {clip} --n-images 32")
        run(f"{E1} extract-clip --clip {clip}")
        run(f"{E1} ground --train-seed {TRAIN_SEED} --k-mode known --cluster-seed 0 --clip {clip}")


step("prepare_data", prepare)
step("weights", weights)
if RUN_BENCHMARK:
    step("benchmark", bench, needs=["prepare_data", "weights"])
step("train", train, needs=["prepare_data", "weights"])
if finished():
    step("downstream", downstream, needs=["prepare_data"])
else:
    STATUS["downstream"] = "waiting: training not finished (attach this version's output and run again)"
step("report", lambda: run(f"{E1} report"))

## 8. Status, report and results archive

In [ ]:
(RUNS / DATASET / "NOTEBOOK_STATUS.json").write_text(json.dumps({"commit": COMMIT, "status": STATUS,
                                                               "train_seed": TRAIN_SEED}, indent=2))
# small download: everything except checkpoints, CLIP shards and image matrices (those stay in this
# version's output for the next session and for the mean-shift experiment)
zip_path = WORK / f"e1_results_{DATASET}.zip"
if zip_path.exists():
    zip_path.unlink()
run(f"cd {RUNS} && zip -qr {zip_path} {DATASET} -x '*/checkpoints/*' '*/image_shards/*' '*/R_*.npy' "
    f"'*/features/*.npz' '*/images/*'", cwd=RUNS)
report = RUNS / DATASET / "report" / "BASELINE_REPORT.md"
if report.exists():
    from IPython.display import Markdown, display
    display(Markdown(report.read_text()))
print("=" * 60)
for k, v in STATUS.items():
    print(f"{k:14s} {v}")
print("=" * 60)